In [6]:
import os
from tqdm import tqdm
import pandas as pd
import subprocess
import pickle
import ast
import numpy as np
import matplotlib.pyplot as plt
import pyts.image as pti
from collections import Counter

import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri

import math
from sklearn.metrics import r2_score
from scipy.interpolate import interp1d
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

from specio import specread
from specio import help
from specio.datasets import load_sp_path

import pandas as pd
import os

In [7]:
classification = pd.read_csv('spectral_matching_classification.csv')
classification.head()

,File name,Color,Polymer,Matching
0,Adv1.1_3.csv,NaN,PE,0.98
1,Adv1.1_4.csv,white,PE,0.95
2,Adv1.1_5.csv,transparent,PE,0.92
3,Adv1.1_6.csv,blue,PP,0.96
4,Adv1.1_7.csv,black,PE,0.98


In [8]:
csv_folder_path = '../1_converting_sp_to_csv/csv_files'

In [9]:
# Get all CSV file names in the folder
csv_files = [f for f in os.listdir(csv_folder_path) if f.endswith('.csv')]

# Iterate over each CSV file
dataframes = []
for csv_file in tqdm(csv_files):
    file_path = os.path.join(csv_folder_path, csv_file)
    df = pd.read_csv(file_path)
    df = df.rename(columns={'Absorbance': csv_file})
    #print(csv_file)
    dataframes.append(df)

100%|██████████████████████████████████████| 2010/2010 [00:03<00:00, 524.13it/s]


In [10]:
# Merge the DataFrames based on a common column "Wavelength"
merged_df = dataframes[0]  # Initialize with the first DataFrame
for df in tqdm(dataframes[1:]):
    merged_df = pd.merge(merged_df, df, on='Wavelength', how='inner')

merged_df

100%|███████████████████████████████████████| 2009/2009 [00:37<00:00, 54.06it/s]


,Wavelength,AI58_6_243.csv,AI58_7_127.csv,Adv_1.2_12.csv,Adv_1.3_35.csv,AI58_8_63.csv,AI58_5_227.csv,AI58_5.2_168.csv,AI58_5_76.csv,AI58_5.2_383.csv,...,AMK86_7275_1.csv,AMK86_7223_5.csv,AI58_5.2_275.csv,AI58_5.2_22.csv,AI58_5.2_96.csv,AI58_35_43.csv,Adv1.1_10.csv,AI58_5.2_251.csv,AI58_6_322.1.csv,AI58_5.2_361.csv
0,4000.0,0.004396,0.001306,0.001579,0.005149,0.017350,0.000783,0.005001,0.003836,0.002218,...,0.008329,0.003385,0.000738,0.008143,0.002888,0.001718,0.006269,0.001876,0.000061,0.002851
1,3999.0,0.004372,0.001326,0.001615,0.005151,0.017324,0.000794,0.005002,0.003859,0.002195,...,0.008354,0.003398,0.000738,0.008188,0.002870,0.001755,0.006295,0.001834,0.000066,0.002859
2,3998.0,0.004368,0.001338,0.001643,0.005153,0.017297,0.000792,0.005004,0.003846,0.002159,...,0.008370,0.003405,0.000737,0.008231,0.002870,0.001859,0.006291,0.001808,0.000048,0.002872
3,3997.0,0.004385,0.001351,0.001668,0.005187,0.017271,0.000779,0.005005,0.003820,0.002130,...,0.008377,0.003406,0.000737,0.008265,0.002892,0.002013,0.006265,0.001807,0.000046,0.002883
4,3996.0,0.004409,0.001365,0.001689,0.005239,0.017260,0.000763,0.005045,0.003795,0.002126,...,0.008382,0.003408,0.000733,0.008281,0.002937,0.002168,0.006237,0.001866,0.000041,0.002887
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3546,454.0,0.022335,0.032976,0.044029,0.029782,0.033289,0.027618,0.113535,0.014868,0.024167,...,0.084017,0.041477,0.042581,0.100562,0.027297,0.025628,0.032463,0.064146,0.051829,0.029951
3547,453.0,0.022636,0.032250,0.043270,0.030592,0.034035,0.026987,0.112124,0.015184,0.023708,...,0.083332,0.041654,0.042552,0.099874,0.027425,0.028293,0.032169,0.064913,0.050393,0.029349
3548,452.0,0.022554,0.030956,0.040748,0.030948,0.035539,0.027380,0.110051,0.016428,0.022751,...,0.082693,0.041722,0.042867,0.098193,0.026886,0.026461,0.031982,0.065575,0.050630,0.028812
3549,451.0,0.021567,0.029051,0.038374,0.030784,0.037085,0.026828,0.107882,0.017438,0.021574,...,0.082256,0.041720,0.043395,0.095510,0.026150,0.023995,0.032060,0.065426,0.051949,0.028258


In [11]:
merged_df_T = merged_df.T.reset_index()
merged_df_T.columns = merged_df_T.iloc[0]
merged_df_T = merged_df_T[1:]
merged_df_T = merged_df_T.rename(columns={'Wavelength': "File name"})
merged_df_T

,File name,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,3994.0,3993.0,3992.0,...,459.0,458.0,457.0,456.0,455.0,454.0,453.0,452.0,451.0,450.0
1,AI58_6_243.csv,0.004396,0.004372,0.004368,0.004385,0.004409,0.004433,0.004451,0.004462,0.004467,...,0.021936,0.025098,0.024264,0.023139,0.022402,0.022335,0.022636,0.022554,0.021567,0.020142
2,AI58_7_127.csv,0.001306,0.001326,0.001338,0.001351,0.001365,0.001386,0.001402,0.001404,0.001392,...,0.034093,0.035143,0.035334,0.034582,0.033659,0.032976,0.032250,0.030956,0.029051,0.027053
3,Adv_1.2_12.csv,0.001579,0.001615,0.001643,0.001668,0.001689,0.001704,0.001704,0.001677,0.001626,...,0.040879,0.040895,0.042254,0.043069,0.043565,0.044029,0.043270,0.040748,0.038374,0.038164
4,Adv_1.3_35.csv,0.005149,0.005151,0.005153,0.005187,0.005239,0.005286,0.005305,0.005289,0.005251,...,0.031766,0.031298,0.030798,0.029933,0.029349,0.029782,0.030592,0.030948,0.030784,0.030382
5,AI58_8_63.csv,0.017350,0.017324,0.017297,0.017271,0.017260,0.017270,0.017266,0.017282,0.017294,...,0.033486,0.034703,0.034774,0.034003,0.033328,0.033289,0.034035,0.035539,0.037085,0.037905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2006,AI58_35_43.csv,0.001718,0.001755,0.001859,0.002013,0.002168,0.001556,0.001590,0.001614,0.001626,...,0.028465,0.029154,0.028936,0.027824,0.026223,0.025628,0.028293,0.026461,0.023995,0.022004
2007,Adv1.1_10.csv,0.006269,0.006295,0.006291,0.006265,0.006237,0.006219,0.006207,0.006194,0.006186,...,0.034558,0.034732,0.034405,0.033643,0.032925,0.032463,0.032169,0.031982,0.032060,0.032411
2008,AI58_5.2_251.csv,0.001876,0.001834,0.001808,0.001807,0.001866,0.001928,0.001994,0.002046,0.002078,...,0.067603,0.067390,0.066574,0.065211,0.064165,0.064146,0.064913,0.065575,0.065426,0.064318
2009,AI58_6_322.1.csv,0.000061,0.000066,0.000048,0.000046,0.000041,0.000046,0.000067,0.000097,0.000123,...,0.054975,0.054200,0.055414,0.055430,0.054052,0.051829,0.050393,0.050630,0.051949,0.053102


In [12]:
# Merge df2 into df1 based on the common_column
our_database = pd.merge(classification, merged_df_T, on='File name', how='left')

In [13]:
our_database

,File name,Color,Polymer,Matching,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,...,459.0,458.0,457.0,456.0,455.0,454.0,453.0,452.0,451.0,450.0
0,Adv1.1_3.csv,NaN,PE,0.98,0.001491,0.001483,0.001483,0.001492,0.001509,0.001532,...,0.023790,0.027825,0.032067,0.032876,0.030447,0.027955,0.027342,0.028074,0.028446,0.027935
1,Adv1.1_4.csv,white,PE,0.95,0.002901,0.002914,0.002915,0.002900,0.002873,0.002843,...,0.056986,0.056147,0.054138,0.051779,0.050458,0.050193,0.049825,0.048727,0.047823,0.047667
2,Adv1.1_5.csv,transparent,PE,0.92,0.002795,0.002797,0.002800,0.002804,0.002808,0.002809,...,0.097227,0.100712,0.101276,0.097796,0.091534,0.085392,0.081877,0.082037,0.083644,0.084670
3,Adv1.1_6.csv,blue,PP,0.96,0.001478,0.001477,0.001489,0.001509,0.001533,0.001553,...,0.018965,0.019365,0.019543,0.018910,0.018044,0.017325,0.016312,0.014865,0.013980,0.013988
4,Adv1.1_7.csv,black,PE,0.98,0.008695,0.008685,0.008666,0.008636,0.008603,0.008585,...,0.043483,0.044460,0.044129,0.043891,0.045927,0.049857,0.052803,0.052295,0.049565,0.047077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2005,Templfj_3_35.csv,white,PP,0.90,0.006937,0.006916,0.006880,0.006831,0.006785,0.006760,...,0.017704,0.022638,0.029100,0.033103,0.033064,0.028990,0.022391,0.016495,0.015351,0.018589
2006,Templfj_3_36.csv,yellow,PE,0.95,0.000878,0.000866,0.000851,0.000841,0.000841,0.000846,...,0.030034,0.029371,0.028058,0.026900,0.026368,0.026341,0.026481,0.026589,0.026619,0.026577
2007,Templfj_3_37.csv,white,PS,0.96,0.025721,0.025700,0.025672,0.025616,0.025580,0.025550,...,0.082940,0.081200,0.079504,0.078298,0.077113,0.075806,0.074342,0.073169,0.072596,0.072377
2008,Templfj_3_38.csv,white,PS,0.93,0.007784,0.007787,0.007788,0.007790,0.007794,0.007800,...,0.023619,0.023117,0.022551,0.021263,0.019810,0.018438,0.017704,0.017871,0.018501,0.018942


In [16]:
our_database.to_csv('our_database_12_03_2025_red.csv', index=False)

In [17]:
our_database

,File name,Color,Polymer,Matching,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,...,459.0,458.0,457.0,456.0,455.0,454.0,453.0,452.0,451.0,450.0
0,Adv1.1_3.csv,NaN,PE,0.98,0.001491,0.001483,0.001483,0.001492,0.001509,0.001532,...,0.023790,0.027825,0.032067,0.032876,0.030447,0.027955,0.027342,0.028074,0.028446,0.027935
1,Adv1.1_4.csv,white,PE,0.95,0.002901,0.002914,0.002915,0.002900,0.002873,0.002843,...,0.056986,0.056147,0.054138,0.051779,0.050458,0.050193,0.049825,0.048727,0.047823,0.047667
2,Adv1.1_5.csv,transparent,PE,0.92,0.002795,0.002797,0.002800,0.002804,0.002808,0.002809,...,0.097227,0.100712,0.101276,0.097796,0.091534,0.085392,0.081877,0.082037,0.083644,0.084670
3,Adv1.1_6.csv,blue,PP,0.96,0.001478,0.001477,0.001489,0.001509,0.001533,0.001553,...,0.018965,0.019365,0.019543,0.018910,0.018044,0.017325,0.016312,0.014865,0.013980,0.013988
4,Adv1.1_7.csv,black,PE,0.98,0.008695,0.008685,0.008666,0.008636,0.008603,0.008585,...,0.043483,0.044460,0.044129,0.043891,0.045927,0.049857,0.052803,0.052295,0.049565,0.047077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2005,Templfj_3_35.csv,white,PP,0.90,0.006937,0.006916,0.006880,0.006831,0.006785,0.006760,...,0.017704,0.022638,0.029100,0.033103,0.033064,0.028990,0.022391,0.016495,0.015351,0.018589
2006,Templfj_3_36.csv,yellow,PE,0.95,0.000878,0.000866,0.000851,0.000841,0.000841,0.000846,...,0.030034,0.029371,0.028058,0.026900,0.026368,0.026341,0.026481,0.026589,0.026619,0.026577
2007,Templfj_3_37.csv,white,PS,0.96,0.025721,0.025700,0.025672,0.025616,0.025580,0.025550,...,0.082940,0.081200,0.079504,0.078298,0.077113,0.075806,0.074342,0.073169,0.072596,0.072377
2008,Templfj_3_38.csv,white,PS,0.93,0.007784,0.007787,0.007788,0.007790,0.007794,0.007800,...,0.023619,0.023117,0.022551,0.021263,0.019810,0.018438,0.017704,0.017871,0.018501,0.018942
